# Entrenamiento y validación

Primero solo voy a usar los datos de entrenamiento para que los modelos aprendan. Después los evaluare con el conjunto de validación y voy a seleccioniar el que obtenga mejores resultados. Los datos de prueba final se mantendrán separados hasta el final para comprobar cómo funciona el modelo elegido

In [ ]:
from pathlib import Path

import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Localizo la carpeta principal
ruta_actual = Path.cwd().resolve()

if ruta_actual.name == "notebooks":
    ruta_proyecto = ruta_actual.parent
else:
    ruta_proyecto = ruta_actual

archivo_particiones = (
    ruta_proyecto
    / "data"
    / "processed"
    / "eurusd_yahoo_particiones.csv"
)

datos = pd.read_csv(
    archivo_particiones,
    parse_dates=["Date"],
    index_col="Date"
)

datos["Objetivo"] = datos["Objetivo"].astype("Int64")

print("Dimensiones:", datos.shape)

Dimensiones: (5825, 21)


In [2]:
# Variables que realmente utilizarán los modelos
variables_predictoras = [
    "Retorno_diario",
    "Retorno_lag_1",
    "Retorno_lag_2",
    "Retorno_lag_3",
    "Retorno_lag_5",
    "Rango_diario",
    "Cuerpo_vela",
    "Posicion_cierre",
    "Distancia_MA5",
    "Distancia_MA10",
    "Distancia_MA20",
    "Volatilidad_5",
    "Volatilidad_20",
    "RSI_14",
    "MACD_hist"
]
#Los indicadores ya los calculé, aquí solo especifico cuáles columnas utilizarán los modelos como datos de entrada

# Separo las dos particiones que usaré por ahora
entrenamiento = datos[
    datos["Particion"] == "entrenamiento"
].copy()

validacion = datos[
    datos["Particion"] == "validacion"
].copy()

X_entrenamiento = entrenamiento[variables_predictoras]
y_entrenamiento = entrenamiento["Objetivo"].astype(int)

X_validacion = validacion[variables_predictoras]
y_validacion = validacion["Objetivo"].astype(int)

print("Entrenamiento:", X_entrenamiento.shape)
print("Validación:", X_validacion.shape)

Entrenamiento: (4908, 15)
Validación: (522, 15)


In [3]:
# Creo una función con los pasos comunes de entrenamiento y evaluación para aplicarlos de la misma manera a cada modelo
# mas que nada para no repetir codigo

def calcular_metricas(
    nombre, # del modelo
    y_real,
    y_predicho,
    probabilidades=None # Algunos modelos no solo indican si la siguiente jornada subirá o bajará, sino que también muestran qué tan segura es 
                        # esa predicción mediante una probabilidad. Otros modelos simplemente entregan la clase final (0 o 1), sin probabilidad
                        # por eso dejo este dato como opcional, cuando el modelo proporciona probabilidades calculo también el ROC AUC
):
    resultado = {
        "Modelo": nombre,
        "Accuracy": accuracy_score(
            y_real,
            y_predicho
        ),
        "Balanced_accuracy": balanced_accuracy_score(
            y_real,
            y_predicho
        ),
        "Precision": precision_score(
            y_real,
            y_predicho,
            zero_division=0
        ),
        "Recall": recall_score(
            y_real,
            y_predicho,
            zero_division=0
        ),
        "F1": f1_score(
            y_real,
            y_predicho,
            zero_division=0
        ),
        "ROC_AUC": None
    }

    if probabilidades is not None:
        resultado["ROC_AUC"] = roc_auc_score(
            y_real,
            probabilidades
        )

    return resultado

# MODELO BASE: CLASE MAYORITARIA

In [4]:
modelo_mayoria = DummyClassifier( # Este modelo utiliza como predicción la clase que aparece con mayor frecuencia en los datos de entrenamiento
                                  # No analiza las variables predictoras, sino que sirve como referencia mínima para comprobar si los modelos 
                                  # posteriores realmente aprenden algo útil (este modelo es lo minimo que se espera)
    strategy="most_frequent"
)

modelo_mayoria.fit(
    X_entrenamiento,
    y_entrenamiento
)

pred_mayoria = modelo_mayoria.predict(
    X_validacion
)

prob_mayoria = modelo_mayoria.predict_proba(
    X_validacion
)[:, 1]

resultados = [] # una lista vacía para guardar las métricas de todos los modelos

resultados.append(
    calcular_metricas(
        "Clase mayoritaria",
        y_validacion,
        pred_mayoria,
        prob_mayoria
    )
)

# Modelo base de persistencia

In [5]:
pred_persistencia = (
    validacion["Retorno_diario"] > 0
).astype(int)

resultados.append(
    calcular_metricas(
        "Persistencia",
        y_validacion,
        pred_persistencia
    )
)

# Este es un modelo demasiado sencillo (pero sirve para comprobar si los algoritmos superan una regla básica basada en el movimiento reciente)

# Regresión logística

In [6]:
# este ya es el primer modelo real

modelo_logistico = Pipeline([ # Utilizo un Pipeline para mantener unidos el escalado de las variables y el entrenamiento de la regresión logística 
                              # Como los indicadores tienen escalas muy diferentes primero se transforman a valores comparables y después se 
                              # entregan al modelo, ademas el escalador aprende únicamente con los datos de entrenamiento y aplica esa misma 
                              # transformación a validación, evitando utilizar información del futuro
    (
        "escalador", # utilizo el mismo criterio de escalado en la validacion
        StandardScaler()
    ),
    (
        "modelo",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

modelo_logistico.fit(
    X_entrenamiento,
    y_entrenamiento
)

# Para convertir las probabilidades en una predicción final se mantiene el límite estándar de 0,50. Si la probabilidad de subida es igual o 
# superior a 0,50, se asigna la clase 1, en caso contrario, se asigna la clase 0. Como ambas clases aparecen en proporciones muy similares, 
# no es necesario modificar este límite para favorecer una sobre la otra.

pred_logistica = modelo_logistico.predict(
    X_validacion
)

prob_logistica = modelo_logistico.predict_proba(
    X_validacion
)[:, 1]

resultados.append(
    calcular_metricas(
        "Regresión logística",
        y_validacion,
        pred_logistica,
        prob_logistica
    )
)

In [ ]:
resultados_validacion = pd.DataFrame( # quiero ver si se está llenando correctamente la tabla de resultados y tambien quiero comprobar
                                      # si a la regresion logistica le fue mejor que los modelos base antes de hacer modelos mas complejos
    resultados
).set_index("Modelo")

resultados_validacion = (
    resultados_validacion
    .astype(float)
    .round(4)
)

display(resultados_validacion)

,Accuracy,Balanced_accuracy,Precision,Recall,F1,ROC_AUC
Modelo,,,,,,
Clase mayoritaria,0.5172,0.5000,0.0000,0.0000,0.0000,0.5000
Persistencia,0.5307,0.5302,0.5138,0.5159,0.5149,NaN
Regresión logística,0.7950,0.7951,0.7821,0.7976,0.7898,0.8716
